In [195]:
import os
import json
import pandas as pd
import numpy as np

In [196]:
ledger_path = "01_data/raw/ledger.csv"
gateway_path = "01_data/raw/gateway.csv"

ledger = pd.read_csv(ledger_path)
gateway = pd.read_csv(gateway_path)

print("Ledger Shape:", ledger.shape)
print("Gateway Shape:", gateway.shape)

Ledger Shape: (10, 7)
Gateway Shape: (9, 7)


In [197]:
print("Ledger Sample")
display(ledger.head())

print("Gateway Sample")
display(gateway.head())

Ledger Sample


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,recon_status
0,R001,01-03-2026,M001,1200,success,UPI,NaN
1,R002,01-03-2026,M002,850,success,Card,NaN
2,R003,02-03-2026,M001,500,success,Wallet,NaN
3,R004,02-03-2026,M003,2100,success,Card,NaN
4,R005,03-03-2026,M004,7200,success,Card,NaN


Gateway Sample


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,recon_status
0,R001,01-03-2026,M001,1200,success,UPI,NaN
1,R002,01-03-2026,M002,900,success,Card,NaN
2,R003,02-03-2026,M001,500,success,Wallet,NaN
3,R005,03-03-2026,M004,7200,failed,Card,NaN
4,R006,03-03-2026,M002,950,success,UPI,NaN


In [198]:
print("Ledger Columns")
print(ledger.columns)

print("\nGateway Columns")
print(gateway.columns)

Ledger Columns
Index(['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd',
       'status', 'payment_method', 'recon_status'],
      dtype='object')

Gateway Columns
Index(['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd',
       'status', 'payment_method', 'recon_status'],
      dtype='object')


In [199]:
print("Ledger Null Values")
print(ledger.isnull().sum())

print("\nGateway Null Values")
print(gateway.isnull().sum())

Ledger Null Values
transaction_id       0
transaction_date     0
merchant_id          0
amount_usd           0
status               0
payment_method       0
recon_status        10
dtype: int64

Gateway Null Values
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
recon_status        9
dtype: int64


In [200]:
ledger_duplicates = ledger[
    ledger.duplicated(subset=["transaction_id"], keep=False)
]

gateway_duplicates = gateway[
    gateway.duplicated(subset=["transaction_id"], keep=False)
]

print("Ledger Duplicate Count:", len(ledger_duplicates))
print("Gateway Duplicate Count:", len(gateway_duplicates))

Ledger Duplicate Count: 0
Gateway Duplicate Count: 0


In [201]:
display(ledger_duplicates)
display(gateway_duplicates)

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,recon_status


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,recon_status


In [202]:
recon = pd.merge(
    ledger,
    gateway,
    on="transaction_id",
    how="outer",
    suffixes=("_ledger", "_gateway"),
    indicator=True
)

print(recon.head())

  transaction_id transaction_date_ledger merchant_id_ledger  \
0           R001              01-03-2026               M001   
1           R002              01-03-2026               M002   
2           R003              02-03-2026               M001   
3           R004              02-03-2026               M003   
4           R005              03-03-2026               M004   

   amount_usd_ledger status_ledger payment_method_ledger  recon_status_ledger  \
0             1200.0       success                   UPI                  NaN   
1              850.0       success                  Card                  NaN   
2              500.0       success                Wallet                  NaN   
3             2100.0       success                  Card                  NaN   
4             7200.0       success                  Card                  NaN   

  transaction_date_gateway merchant_id_gateway  amount_usd_gateway  \
0               01-03-2026                M001              1200

In [245]:
import pandas as pd

# Load cleaned transactions
cleaned_transactions = pd.read_csv(
    "01_data/processed/cleaned_transactions.csv"
)

# Convert important columns
cleaned_transactions["amount_usd"] = pd.to_numeric(
    cleaned_transactions["amount_usd"],
    errors="coerce"
)

cleaned_transactions["risk_score"] = (
    cleaned_transactions["risk_score"]
    .astype(str)
    .str.extract(r"(\d+)")
)

cleaned_transactions["risk_score"] = pd.to_numeric(
    cleaned_transactions["risk_score"],
    errors="coerce"
)

cleaned_transactions["high_risk_flag"] = pd.to_numeric(
    cleaned_transactions["high_risk_flag"],
    errors="coerce"
)

cleaned_transactions["high_value_flag"] = pd.to_numeric(
    cleaned_transactions["high_value_flag"],
    errors="coerce"
)

# Create chargeback flag
cleaned_transactions["chargeback_flag"] = (
    cleaned_transactions["status"]
    .astype(str)
    .str.lower()
    .str.contains("chargeback")
    .astype(int)
)

# Merchant Risk Summary
merchant_risk_summary = cleaned_transactions.groupby(
    "merchant_name"
).agg(
    total_transactions=("transaction_id", "count"),
    total_gmv=("amount_usd", "sum"),
    avg_risk_score=("risk_score", "mean"),
    high_risk_transactions=("high_risk_flag", "sum"),
    high_value_transactions=("high_value_flag", "sum"),
    chargeback_transactions=("chargeback_flag", "sum")
).reset_index()

# Round numeric values
merchant_risk_summary["avg_risk_score"] = (
    merchant_risk_summary["avg_risk_score"]
    .round(2)
)

merchant_risk_summary["total_gmv"] = (
    merchant_risk_summary["total_gmv"]
    .round(2)
)

# Sort by GMV descending
merchant_risk_summary = merchant_risk_summary.sort_values(
    by="total_gmv",
    ascending=False
)

# Export CSV
merchant_risk_summary.to_csv(
    "01_data/processed/merchant_risk_summary.csv",
    index=False
)

print("merchant_risk_summary.csv created successfully.")

merchant_risk_summary.csv created successfully.


In [203]:
missing_in_gateway = recon[
    recon["_merge"] == "left_only"
]

print("Records Missing in Gateway:", len(missing_in_gateway))

display(missing_in_gateway)

Records Missing in Gateway: 2


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,recon_status_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,recon_status_gateway,_merge
3,R004,02-03-2026,M003,2100.0,success,Card,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
9,R010,05-03-2026,M004,2500.0,success,Wallet,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [204]:
missing_in_ledger = recon[
    recon["_merge"] == "right_only"
]

print("Records Missing in Ledger:", len(missing_in_ledger))

display(missing_in_ledger)

Records Missing in Ledger: 1


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,recon_status_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,recon_status_gateway,_merge
10,R011,NaN,NaN,NaN,NaN,NaN,NaN,05-03-2026,M003,1800.0,success,Card,NaN,right_only


In [205]:
amount_mismatch = recon[
    (recon["_merge"] == "both") &
    (
        recon["amount_usd_ledger"] !=
        recon["amount_usd_gateway"]
    )
]

print("Amount Mismatch Count:", len(amount_mismatch))

display(amount_mismatch.head())

Amount Mismatch Count: 2


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,recon_status_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,recon_status_gateway,_merge
1,R002,01-03-2026,M002,850.0,success,Card,NaN,01-03-2026,M002,900.0,success,Card,NaN,both
7,R008,04-03-2026,M001,640.0,success,Card,NaN,04-03-2026,M001,600.0,success,Card,NaN,both


In [206]:
status_mismatch = recon[
    (recon["_merge"] == "both") &
    (
        recon["status_ledger"] !=
        recon["status_gateway"]
    )
]

print("Status Mismatch Count:", len(status_mismatch))

display(status_mismatch.head())

Status Mismatch Count: 1


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,recon_status_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,recon_status_gateway,_merge
4,R005,03-03-2026,M004,7200.0,success,Card,NaN,03-03-2026,M004,7200.0,failed,Card,NaN,both


In [207]:
recon["recon_status"] = np.where(
    recon["_merge"] == "left_only",
    "Missing in Gateway",

    np.where(
        recon["_merge"] == "right_only",
        "Missing in Ledger",

        np.where(
            recon["amount_usd_ledger"] !=
            recon["amount_usd_gateway"],
            "Amount Mismatch",

            np.where(
                recon["status_ledger"] !=
                recon["status_gateway"],
                "Status Mismatch",
                "Matched"
            )
        )
    )
)

display(recon.head())

,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,recon_status_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,recon_status_gateway,_merge,recon_status
0,R001,01-03-2026,M001,1200.0,success,UPI,NaN,01-03-2026,M001,1200.0,success,UPI,NaN,both,Matched
1,R002,01-03-2026,M002,850.0,success,Card,NaN,01-03-2026,M002,900.0,success,Card,NaN,both,Amount Mismatch
2,R003,02-03-2026,M001,500.0,success,Wallet,NaN,02-03-2026,M001,500.0,success,Wallet,NaN,both,Matched
3,R004,02-03-2026,M003,2100.0,success,Card,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Missing in Gateway
4,R005,03-03-2026,M004,7200.0,success,Card,NaN,03-03-2026,M004,7200.0,failed,Card,NaN,both,Status Mismatch


In [208]:
summary_metrics = {

    "total_ledger_rows": int(len(ledger)),

    "total_gateway_rows": int(len(gateway)),

    "missing_in_gateway_count": int(
        len(missing_in_gateway)
    ),

    "missing_in_ledger_count": int(
        len(missing_in_ledger)
    ),

    "amount_mismatch_count": int(
        len(amount_mismatch)
    ),

    "status_mismatch_count": int(
        len(status_mismatch)
    ),

    "reconciliation_issue_count": int(
        len(missing_in_gateway)
        +
        len(missing_in_ledger)
        +
        len(amount_mismatch)
        +
        len(status_mismatch)
    ),

    "ledger_total_amount": float(
        pd.to_numeric(
            ledger["amount_usd"],
            errors="coerce"
        ).sum()
    ),

    "gateway_total_amount": float(
        pd.to_numeric(
            gateway["amount_usd"],
            errors="coerce"
        ).sum()
    ),

    "amount_at_risk": float(
        pd.to_numeric(
            amount_mismatch["amount_usd_ledger"],
            errors="coerce"
        ).sum()
    )
}

print(summary_metrics)

{'total_ledger_rows': 10, 'total_gateway_rows': 9, 'missing_in_gateway_count': 2, 'missing_in_ledger_count': 1, 'amount_mismatch_count': 2, 'status_mismatch_count': 1, 'reconciliation_issue_count': 6, 'ledger_total_amount': 23340.0, 'gateway_total_amount': 20550.0, 'amount_at_risk': 1490.0}


In [209]:
os.makedirs("04_python", exist_ok=True)

with open(
    "04_python/summary_metrics.json",
    "w"
) as json_file:
    json.dump(summary_metrics, json_file, indent=4)

print("summary_metrics.json saved successfully.")

summary_metrics.json saved successfully.


In [210]:
os.makedirs("01_data/processed", exist_ok=True)

In [211]:
missing_in_gateway.to_csv(
    "01_data/processed/missing_in_gateway.csv",
    index=False
)

In [212]:
missing_in_ledger.to_csv(
    "01_data/processed/missing_in_ledger.csv",
    index=False
)

In [213]:
amount_mismatch.to_csv(
    "01_data/processed/amount_mismatches.csv",
    index=False
)

In [214]:
status_mismatch.to_csv(
"01_data/processed/status_mismatches.csv",
index=False
)

In [215]:
os.makedirs("01_data/processed", exist_ok=True)

recon.to_csv(
    "01_data/processed/reconciliation_report.csv",
    index=False
)

print("reconciliation_report.csv saved successfully.")

reconciliation_report.csv saved successfully.


In [216]:
import os

os.makedirs(
    "01_data/processed",
    exist_ok=True
)

print("Processed folder ready.")

Processed folder ready.


In [217]:
import pandas as pd
import json

json_path = "01_data/raw/api_response_sample.json"

with open(json_path, "r") as file:
    data = json.load(file)

print(data)

{'generated_at': '2026-03-07T10:00:00Z', 'source': 'QuickPay Settlement API', 'batches': [{'batch_id': 'B001', 'merchant': {'merchant_id': 'M001', 'merchant_name': 'Alpha Mart', 'region': 'APAC'}, 'settlements': [{'settlement_id': 'S001', 'amount_usd': 1520.5, 'status': 'settled', 'processed_at': '2026-03-07T08:10:00Z', 'bank': {'name': 'Bank A', 'country': 'IN'}}, {'settlement_id': 'S002', 'amount_usd': 980.0, 'status': 'pending', 'processed_at': '2026-03-07T08:45:00Z', 'bank': {'name': 'Bank A', 'country': 'IN'}}, {'settlement_id': 'S003', 'amount_usd': 640.0, 'status': 'settled', 'processed_at': '2026-03-07T09:15:00Z', 'bank': {'name': 'Bank B', 'country': 'SG'}}]}, {'batch_id': 'B002', 'merchant': {'merchant_id': 'M004', 'merchant_name': 'Delta Travels', 'region': 'US'}, 'settlements': [{'settlement_id': 'S004', 'amount_usd': 2100.0, 'status': 'settled', 'processed_at': '2026-03-07T08:20:00Z', 'bank': {'name': 'Bank C', 'country': 'US'}}, {'settlement_id': 'S005', 'amount_usd': 500

In [218]:
normalized_df = pd.json_normalize(data)

print(normalized_df.head())

           generated_at                   source  \
0  2026-03-07T10:00:00Z  QuickPay Settlement API   

                                             batches  
0  [{'batch_id': 'B001', 'merchant': {'merchant_i...  


In [220]:
print(normalized_df.columns)

Index(['generated_at', 'source', 'batches'], dtype='object')


In [221]:
normalized_df["generated_at"] = pd.to_datetime(
    normalized_df["generated_at"]
)

In [222]:
normalized_df.to_csv(
    "01_data/processed/api_normalized.csv",
    index=False
)

print("api_normalized.csv exported.")

api_normalized.csv exported.


In [223]:
cleaned_transactions = pd.read_csv(
    "01_data/processed/cleaned_transactions.csv"
)

In [224]:
cleaned_transactions["transaction_date"] = pd.to_datetime(
    cleaned_transactions["transaction_date"]
)

In [225]:
daily_summary = cleaned_transactions.groupby(
    cleaned_transactions["transaction_date"].dt.date
).agg(
    total_gmv=("amount_usd", "sum"),
    total_transactions=("transaction_id", "count"),
    successful_transactions=(
        "status",
        lambda x: (x == "success").sum()
    )
).reset_index()

In [226]:
daily_summary.columns = [
    "date",
    "total_gmv",
    "total_transactions",
    "successful_transactions"
]

In [227]:
daily_summary.to_csv(
    "01_data/processed/daily_summary.csv",
    index=False
)

print("daily_summary.csv exported.")

daily_summary.csv exported.


In [228]:
payment_method_breakdown = cleaned_transactions.groupby(
    "payment_method"
).agg(
    total_gmv=("amount_usd", "sum"),
    total_transactions=("transaction_id", "count")
).reset_index()

In [229]:
payment_method_breakdown.to_csv(
    "01_data/processed/payment_method_breakdown.csv",
    index=False
)

print("payment_method_breakdown.csv exported.")

payment_method_breakdown.csv exported.


In [230]:
cleaned_transactions["risk_score"] = pd.to_numeric(
    cleaned_transactions["risk_score"],
    errors="coerce"
)

In [231]:
region_breakdown = cleaned_transactions.groupby(
    "gateway_region"
).agg(
    total_gmv=("amount_usd", "sum"),
    total_transactions=("transaction_id", "count"),
    avg_risk_score=("risk_score", "mean")
).reset_index()

In [232]:
region_breakdown["avg_risk_score"] = (
    region_breakdown["avg_risk_score"]
    .round(2)
)

In [233]:
region_breakdown.to_csv(
    "01_data/processed/region_breakdown.csv",
    index=False
)


print("region_breakdown.csv exported.")

region_breakdown.csv exported.


In [234]:
merchant_performance_summary = cleaned_transactions.groupby(
    "merchant_name"
).agg(
    total_gmv=("amount_usd", "sum"),
    total_transactions=("transaction_id", "count"),
    avg_transaction_value=("amount_usd", "mean"),
    high_risk_transactions=("high_risk_flag", "sum"),
    high_value_transactions=("high_value_flag", "sum")
).reset_index()

In [235]:
merchant_performance_summary[
    "avg_transaction_value"
] = merchant_performance_summary[
    "avg_transaction_value"
].round(2)

In [236]:
merchant_performance_summary.to_csv(
    "01_data/processed/merchant_performance_summary.csv",
    index=False
)

print("merchant_performance_summary.csv exported.")

merchant_performance_summary.csv exported.


In [237]:
os.makedirs("04_python", exist_ok=True)

import json

with open(
    "04_python/summary_metrics.json",
    "w"
) as json_file:

    json.dump(
        summary_metrics,
        json_file,
        indent=4
    )

print("summary_metrics.json saved successfully.")

summary_metrics.json saved successfully.


In [238]:
import os

os.makedirs("02_spreadsheet", exist_ok=True)
os.makedirs("03_sql", exist_ok=True)
os.makedirs("05_visualization", exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [239]:

print(os.listdir())

['.config', 'README.md', '04_python', '03_sql', '02_spreadsheet', '05_visualization', '01_data', 'sample_data']


In [240]:
import os

folders = [
    "01_data/raw",
    "01_data/processed",
    "02_spreadsheet",
    "03_sql",
    "04_python",
    "05_visualization"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Repository structure created successfully.")

Repository structure created successfully.


In [241]:
readme_content = """
# QuickPay FinTech Operations Case Study

## Student Information

- Student Name: Mildred Mark Rebello
- Student ID: bitsom_ftai_2601085
- Public GitHub Repository Link: https://github.com/mildredg2608-pixel/QuickPay-FinTech-Ops

---

## Project Overview

This project covers:
- Spreadsheet data cleaning and standardization
- SQL-based business analysis
- Python reconciliation workflows
- JSON normalization using Pandas
- Dashboard visualization using Looker Studio

---

## Repository Structure

- 01_data
  - raw datasets
  - processed outputs

- 02_spreadsheet
  - spreadsheet workbook
  - spreadsheet answers

- 03_sql
  - SQL analysis queries
  - SQL answers

- 04_python
  - reconciliation notebook
  - summary metrics JSON

- 05_visualization
  - dashboard link

---

## Tools Used

- Python
- Pandas
- NumPy
- Jupyter Notebook / Google Colab
- Excel / Google Sheets
- SQL
- Looker Studio

---

## Run Instructions

1. Open the notebook:
   04_python/fintech_pipeline.ipynb

2. Ensure all raw files are placed inside:
   01_data/raw/

3. Run notebook cells sequentially.

4. Processed output files will be generated inside:
   01_data/processed/

5. Dashboard link is available inside:
   05_visualization/dashboard_link.txt

---

## Output Files Generated

- cleaned_transactions.csv
- merchant_risk_summary.csv
- missing_in_gateway.csv
- missing_in_ledger.csv
- amount_mismatches.csv
- status_mismatches.csv
- reconciliation_report.csv
- api_normalized.csv
- daily_summary.csv
- payment_method_breakdown.csv
- region_breakdown.csv
- merchant_performance_summary.csv
- summary_metrics.json

"""

with open("README.md", "w") as file:
    file.write(readme_content)

print("README.md created successfully.")

README.md created successfully.


In [242]:
import os

checkpoint_paths = []

for root, dirs, files in os.walk("."):
    if ".ipynb_checkpoints" in dirs:
        checkpoint_paths.append(
            os.path.join(root, ".ipynb_checkpoints")
        )

if checkpoint_paths:
    print("Found .ipynb_checkpoints folders:\n")

    for path in checkpoint_paths:
        print(path)

else:
    print("No .ipynb_checkpoints folders found.")

No .ipynb_checkpoints folders found.


In [243]:
import os
import shutil

for root, dirs, files in os.walk("."):
    if ".ipynb_checkpoints" in dirs:
        shutil.rmtree(
            os.path.join(root, ".ipynb_checkpoints")
        )

print("All .ipynb_checkpoints directories removed.")

All .ipynb_checkpoints directories removed.
